# DVF 2022 — Origine de l'écart moyenne/médiane du Morbihan

Le département du Morbihan (56) présente le ratio moyenne/médiane le plus élevé du fichier. Ce notebook cherche l'origine de cet écart : il affiche les mutations aux plus fortes valeurs foncières du département, avec le nombre de lignes qu'elles occupent, puis mesure la part de la plus grosse dans la somme totale. La valeur foncière étant répétée sur chaque ligne d'une mutation, une mutation étalée sur de nombreuses lignes pèse lourd dans une moyenne calculée ligne par ligne.

## Cellule 1 — Connexion au fichier

Interrogation avec DuckDB, directement au format Parquet. Le chemin pointe vers le fichier `dvf-2022.parquet` placé dans le dossier `data/`.

In [4]:
import duckdb
from pathlib import Path

# Chemin vers le fichier Parquet, place dans le dossier data/
FICHIER = Path(r"./data/dvf-2022.parquet")

con = duckdb.connect()
pq = str(FICHIER)
assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
print(f"Fichier : {FICHIER.name}")

Fichier : dvf-2022.parquet


## Cellule 2 — Les dix plus fortes valeurs foncières du Morbihan

Les mutations sont approchées par le regroupement (date, commune, valeur foncière), afin de ne pas compter une même vente autant de fois qu'elle a de lignes. La colonne `nb_lignes` indique combien de fois le montant est répété dans le fichier.

In [8]:
top = con.execute(f"""
    SELECT "Date mutation",
           "Commune",
           "Valeur fonciere",
           COUNT(*) AS nb_lignes,
           MAX("Type local") AS type_local_exemple
    FROM '{pq}'
    WHERE "Code departement" = '56'
      AND "Valeur fonciere" IS NOT NULL
    GROUP BY "Date mutation", "Commune", "Valeur fonciere"
    ORDER BY "Valeur fonciere" DESC
    LIMIT 10
""").fetchdf()

top

,Date mutation,Commune,Valeur fonciere,nb_lignes,type_local_exemple
0,2022-12-27,PLOEMEUR,337200416,10,Maison
1,2022-12-27,KERVIGNAC,337200416,2,None
2,2022-12-27,MALGUENAC,337200416,3,None
3,2022-12-27,LOCMIQUELIC,337200416,36,Maison
4,2022-12-27,LANESTER,337200416,7,Maison
5,2022-12-27,BUBRY,337200416,3,Maison
6,2022-12-27,LORIENT,337200416,10046,Maison
7,2022-12-27,GROIX,337200416,32,Local industriel. commercial ou assimilé
8,2022-12-27,QUISTINIC,337200416,25,Maison
9,2022-12-27,PONTIVY,337200416,93,Maison


## Cellule 3 — Poids de la plus grosse mutation

On rapporte la plus grosse mutation (montant multiplié par son nombre de lignes) à la somme des valeurs foncières du département, calculée ligne par ligne. Cette part indique dans quelle mesure une seule vente explique l'écart observé.

In [9]:
somme_dept = con.execute(f"""
    SELECT SUM("Valeur fonciere")
    FROM '{pq}'
    WHERE "Code departement" = '56' AND "Valeur fonciere" IS NOT NULL
""").fetchone()[0]

pg = top.iloc[0]
contrib = pg['Valeur fonciere'] * pg['nb_lignes']
print(f"Somme des valeurs foncieres du Morbihan : {somme_dept:,.0f}".replace(',', ' '))
print(f"Plus grosse mutation, repetee sur {int(pg['nb_lignes'])} lignes : {contrib:,.0f}".replace(',', ' '))
print(f"Part dans la somme du departement : {100 * contrib / somme_dept:.1f} %")

Somme des valeurs foncieres du Morbihan : 9 908 822 129 862
Plus grosse mutation  repetee sur 10 lignes : 3 372 004 160
Part dans la somme du departement : 0.0 %


## Cellule 4 — Poids réel de la mutation dominante

La mutation la plus élevée est étalée sur plusieurs communes : un regroupement par commune la découpe donc en morceaux. Pour mesurer son poids réel, on regroupe ici par valeur foncière seule, ce qui réunit toutes les lignes portant le même montant, puis on rapporte ce total à la somme du département.

In [7]:
par_valeur = con.execute(f"""
    SELECT "Valeur fonciere",
           COUNT(*) AS nb_lignes
    FROM '{pq}'
    WHERE "Code departement" = '56' AND "Valeur fonciere" IS NOT NULL
    GROUP BY "Valeur fonciere"
    ORDER BY "Valeur fonciere" DESC
    LIMIT 5
""").fetchdf()
print(par_valeur.to_string(index=False))

somme_dept = con.execute(f"""
    SELECT SUM("Valeur fonciere")
    FROM '{pq}'
    WHERE "Code departement" = '56' AND "Valeur fonciere" IS NOT NULL
""").fetchone()[0]

v = par_valeur.iloc[0]
contrib = v['Valeur fonciere'] * v['nb_lignes']
print(f"\nMontant dominant : {int(v['Valeur fonciere']):,} EUR, sur {int(v['nb_lignes'])} lignes".replace(',', ' '))
print(f"Part de ce seul montant dans la somme du departement : {100 * contrib / somme_dept:.1f} %")

 Valeur fonciere  nb_lignes
       337200416      14005
       314985152      16411
        17384000          2
        14436000          4
        12444142        142

Montant dominant : 337 200 416 EUR  sur 14005 lignes
Part de ce seul montant dans la somme du departement : 47.7 %
